<a href="https://colab.research.google.com/github/satria1408/saas-microservice1/blob/main/scan_buku_qwen2vl_v3_cache_isbn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Book Scanner - Qwen2-VL (3 Field Inti + Cache SQLite + Jalur ISBN)

Versi ini fokus ke fondasi yang lebih hemat resource:

```
Input: ISBN                          Input: Foto cover
      |                                    |
Lookup Open Library                   Qwen2-VL scan
(TANPA model, langsung                (judul, penulis - penerbit
ke database)                          selalu dari Open Library,
      |                                bukan tebakan Qwen)
      v                                    v
      \------------- SQLite cache ---------/
                       |
         Langsung masuk tabel `katalog` (SQLite)
              status: "otomatis"
                       |
    (opsional) Petugas cek & koreksi lewat
       update_status_konfirmasi -> "terkonfirmasi"
```

Hasil scan TIDAK menunggu konfirmasi petugas untuk masuk database -
begitu selesai diproses, langsung tersimpan ke tabel `katalog`.
Konfirmasi petugas sifatnya opsional, untuk mengoreksi data yang
sudah masuk (bukan syarat sebelum data bisa dipakai).

**3 field inti dulu:** judul, penulis, penerbit. Field lain (ISBN,
edisi, tahun, bahasa, kategori) sengaja belum dimasukkan - fokus benerin
akurasi dasar dulu, baru diperluas belakangan.

**Cache SQLite - 2 lapis:**
1. `cache_scan` - key: hash gambar. Kalau foto YANG SAMA PERSIS pernah
   di-scan sebelumnya, langsung ambil hasil lama, tidak panggil Qwen
   lagi sama sekali (hemat GPU + waktu).
2. `cache_metadata` - key: judul+penulis (dinormalisasi). Kalau ada
   FOTO BEDA dari buku dengan judul yang SAMA, tidak perlu query ulang
   ke Open Library - manfaatkan hasil lookup sebelumnya.

**Jalur ISBN** - kalau petugas sudah punya nomor ISBN (dari cover
belakang/barcode), langsung lookup ke Open Library TANPA melibatkan
Qwen sama sekali - lebih cepat dan lebih presisi (tidak ada ambiguitas
edisi seperti yang sering terjadi lewat foto cover depan saja).

**Dihapus dari versi sebelumnya:** field isbn/edisi/tahun/bahasa/kategori
di prompt scan, dan seluruh bagian Testing/Ground Truth (tidak lagi
dipakai sebagai fondasi awal - foto-foto dataset percobaan sebelumnya
juga sudah tidak relevan, boleh dihapus manual dari Google Drive).

## 1. Setup: Mount Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

FOLDER_PATH = "/content/drive/MyDrive/google-book-scenner"
print("Folder kerja:", FOLDER_PATH)

Mounted at /content/drive
Folder kerja: /content/drive/MyDrive/google-book-scenner


## 2. Setup: Install dependencies

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils pillow pandas
!pip install -q fastapi uvicorn pyngrok nest-asyncio python-multipart requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 38.3 MB/s eta 0:00:00


## 3. Setup: Load model Qwen2-VL-2B-Instruct

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch

MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)

print("Model siap di:", model.device)

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model siap di: cuda:0


In [ ]:
# from huggingface_hub import login
# login(token="isi_token_baru_kamu_di_sini")

## 4. Setup: Cache SQLite (2 tabel)


In [ ]:
import sqlite3
import hashlib
import re

CACHE_DB_PATH = os.path.join(FOLDER_PATH, "book_cache.db")


def _hash_gambar(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def _normalisasi_key(judul: str, penulis: str) -> str:
    gabungan = f"{judul or ''}_{penulis or ''}".lower().strip()
    return re.sub(r"[^a-z0-9]+", "_", gabungan)


def _pastikan_kolom(conn, tabel: str, kolom: str, tipe: str):
    """Cek kolom sudah ada di tabel atau belum, kalau belum, tambahkan.
    Dipakai biar nambah field baru ke depannya cukup daftarin di
    DAFTAR_MIGRASI, tidak perlu tulis blok cek-ALTER manual lagi."""
    kolom_ada = [row[1] for row in conn.execute(f"PRAGMA table_info({tabel})").fetchall()]
    if kolom not in kolom_ada:
        conn.execute(f"ALTER TABLE {tabel} ADD COLUMN {kolom} {tipe}")


# Daftar semua migrasi kolom yang pernah/akan dibutuhkan, di 1 tempat.
# Kalau nanti nambah field baru (misal "edisi", "tahun"), tinggal
# tambah 1 baris di sini.
DAFTAR_MIGRASI = [
    ("cache_scan", "kategori", "TEXT"),
    ("cache_metadata", "isbn", "TEXT"),
    ("katalog", "isbn", "TEXT"),
    ("katalog", "kategori", "TEXT"),
]


def _init_cache_db():
    conn = sqlite3.connect(CACHE_DB_PATH)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS cache_scan (
            hash_gambar TEXT PRIMARY KEY,
            judul TEXT,
            penulis TEXT
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS cache_metadata (
            judul_penulis_key TEXT PRIMARY KEY,
            penerbit TEXT,
            sumber TEXT
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS katalog (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            judul TEXT,
            penulis TEXT,
            penerbit TEXT,
            penerbit_sumber TEXT,
            stok INTEGER DEFAULT 1,
            status_konfirmasi TEXT DEFAULT 'otomatis',
            waktu_masuk TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)

    conn.execute("""
    CREATE TABLE IF NOT EXISTS rag_manual (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        judul TEXT,
        penulis TEXT,
        penerbit TEXT,
        sumber TEXT DEFAULT 'input_manual',
        waktu_masuk TEXT DEFAULT CURRENT_TIMESTAMP
    )
""")

    # Jalankan semua migrasi kolom yang terdaftar
    for tabel, kolom, tipe in DAFTAR_MIGRASI:
        _pastikan_kolom(conn, tabel, kolom, tipe)

    conn.commit()
    conn.close()


_init_cache_db()
print("Cache SQLite siap di:", CACHE_DB_PATH)

Cache SQLite siap di: /content/drive/MyDrive/google-book-scenner/book_cache.db


## 5. Fitur Scan: Prompt (3 field inti, versi ringkas)



In [ ]:
PROMPT = """Lihat sampul buku ini dan ceritakan yang kamu lihat.

Kira-kira apa judul buku ini, siapa penulisnya, siapa penerbitnya
(nama perusahaan penerbit, bukan nama orang), dan buku ini termasuk
kategori/genre apa (misalnya: fiksi, non-fiksi, self-help, akademik,
buku anak/remaja, agama, atau lainnya)?

Kalau ada yang tidak terlihat jelas atau kamu tidak yakin, jawab null
saja untuk bagian itu - itu jawaban yang wajar, tidak masalah.

Jawab dalam format JSON:
{"judul": "...", "penulis": "...", "penerbit": "...", "kategori": "..."}"""

## 6. Fitur Scan: `ask_model` (dengan cache gambar)

.

In [ ]:
import json
import re as _re
from qwen_vl_utils import process_vision_info

CANONICAL_FIELDS = ["judul", "penulis", "penerbit", "kategori"]
KATEGORI_VALID = ["fiksi", "non_fiksi", "self_help", "akademik", "anak_remaja", "agama", "lainnya"]


def _normalisasi_kategori(nilai: str) -> str:
    """Ubah jawaban bebas dari Qwen (misal 'Self-help', 'non fiksi',
    ada typo dsb) jadi salah satu kategori baku, biar konsisten dipakai
    untuk filter/browse nanti - bukan cuma tampilan yang beda-beda."""
    if not nilai:
        return None
    bersih = _re.sub(r"[^a-z]+", "_", nilai.lower()).strip("_")
    for k in KATEGORI_VALID:
        if bersih == k or bersih in k or k in bersih:
            return k
    return "lainnya"


def _normalize_keys(d: dict) -> dict:
    result = {field: None for field in CANONICAL_FIELDS}
    for raw_key, value in d.items():
        clean_key = _re.sub(r"[^a-zA-Z]", "", str(raw_key)).lower()
        for field in CANONICAL_FIELDS:
            if clean_key == field and value not in (None, ""):
                if isinstance(value, str):
                    value = value.strip().strip('"').strip()
                if field == "kategori" and isinstance(value, str):
                    result[field] = _normalisasi_kategori(value)
                else:
                    result[field] = value
    return result


def _generate_once(image_path: str) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=150)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]


def _try_parse_scan(output_text: str):
    cleaned = output_text.strip()
    cleaned = _re.sub(r"^```json", "", cleaned).strip()
    cleaned = _re.sub(r"^```", "", cleaned).strip()
    cleaned = _re.sub(r"```$", "", cleaned).strip()
    cleaned = _re.sub(r"//[^\n]*", "", cleaned)

    match = _re.search(r"\{.*\}", cleaned, _re.DOTALL)
    if match:
        cleaned = match.group(0)

    parsed_raw = None
    try:
        parsed_raw = json.loads(cleaned)
    except json.JSONDecodeError:
        try:
            import json_repair
            parsed_raw = json_repair.loads(cleaned)
        except Exception:
            parsed_raw = None

    if not isinstance(parsed_raw, dict):
        return None

    normalized = _normalize_keys(parsed_raw)
    if not normalized.get("judul") or not normalized.get("penulis"):
        return None

    return normalized


def ask_model(image_path: str, max_retry: int = 2, lengkapi_otomatis: bool = True) -> dict:
    hash_gbr = _hash_gambar(image_path)

    conn = sqlite3.connect(CACHE_DB_PATH)
    row = conn.execute(
        "SELECT judul, penulis, kategori FROM cache_scan WHERE hash_gambar = ?", (hash_gbr,)
    ).fetchone()
    conn.close()

    if row:
        judul, penulis, kategori = row
        parsed = {"judul": judul, "penulis": penulis, "penerbit": None,
                  "kategori": kategori, "_dari_cache_scan": True}
    else:
        last_raw_output = ""
        parsed = None
        for attempt in range(max_retry + 1):
            output_text = _generate_once(image_path)
            last_raw_output = output_text
            hasil = _try_parse_scan(output_text)
            if hasil is not None:
                hasil["_attempt"] = attempt + 1
                hasil["_dari_cache_scan"] = False
                parsed = hasil
                break

        if parsed is None:
            return {"judul": None, "penulis": None, "penerbit": None, "kategori": None,
                    "_raw_output": last_raw_output, "_parse_error": True,
                    "status_konfirmasi": "gagal_scan"}

        conn = sqlite3.connect(CACHE_DB_PATH)
        conn.execute(
            "INSERT OR REPLACE INTO cache_scan (hash_gambar, judul, penulis, kategori) VALUES (?, ?, ?, ?)",
            (hash_gbr, parsed.get("judul"), parsed.get("penulis"), parsed.get("kategori")),
        )
        conn.commit()
        conn.close()

    parsed["_penerbit_tebakan_vlm"] = parsed.get("penerbit")

    if lengkapi_otomatis:
        hasil_meta = lengkapi_metadata(parsed.get("judul"), parsed.get("penulis"))
        parsed["penerbit"] = hasil_meta.get("penerbit")
        parsed["_penerbit_sumber"] = hasil_meta.get("sumber")
        parsed["isbn"] = hasil_meta.get("isbn")
    else:
        parsed["_penerbit_sumber"] = "tidak_dicek"

    parsed["status_konfirmasi"] = "otomatis"
    parsed["_id_katalog"] = _simpan_atau_gabung_katalog(parsed)
    return parsed

## 7. Fitur Scan: Lengkapi Penerbit - Open Library (dengan cache judul)



In [ ]:
import requests
import difflib


def _ambil_dari_open_library(judul: str, penulis: str):
    headers = {"User-Agent": "ScanBukuPerpus/1.0 (contoh@sekolah.sch.id)"}
    BAHASA_PRIORITAS = {"eng", "ind"}

    query = judul
    if penulis:
        query += f" {penulis}"
    search_url = (
        f"https://openlibrary.org/search.json?q={requests.utils.quote(query)}"
        f"&fields=title,author_name,publisher,key&limit=5"
    )

    try:
        res = requests.get(search_url, headers=headers, timeout=15)
        if res.status_code != 200:
            print(f"[DEBUG] Open Library status {res.status_code}")
            return None
        docs = res.json().get("docs", [])
    except Exception as e:
        print(f"[DEBUG] Open Library error: {e}")
        return None

    if not docs:
        return None

    work_key = docs[0].get("key")
    publisher_agregat_fallback = None
    if docs[0].get("publisher"):
        publisher_agregat_fallback = docs[0]["publisher"][0]

    if not work_key:
        return publisher_agregat_fallback

    try:
        ed_url = f"https://openlibrary.org{work_key}/editions.json?limit=20"
        res = requests.get(ed_url, headers=headers, timeout=15)
        if res.status_code != 200:
            return publisher_agregat_fallback
        entries = res.json().get("entries", [])
    except Exception as e:
        print(f"[DEBUG] Open Library editions error: {e}")
        return publisher_agregat_fallback

    kandidat_lain = []
    for ed in entries:
        pub = ed.get("publishers")
        if not pub:
            continue
        lang_keys = [l.get("key", "").split("/")[-1] for l in ed.get("languages", [])]
        if any(l in BAHASA_PRIORITAS for l in lang_keys):
            return pub[0]
        kandidat_lain.append(pub[0])

    if kandidat_lain:
        return kandidat_lain[0]

    if publisher_agregat_fallback:
        print(f"[DEBUG] Tidak ada edisi eng/ind dengan publisher untuk {work_key}, pakai fallback agregat")
    return publisher_agregat_fallback


def tambah_ke_rag_manual(judul: str, penulis: str, penerbit: str, sumber: str = "input_manual"):
    """Simpan data buku yang sudah diverifikasi manual (misal hasil cek
    fisik buku langsung, atau lookup ISBN yang sudah pasti benar) ke
    basis RAG - dipakai sebagai fallback kalau Open Library gagal total."""
    with sqlite3.connect(CACHE_DB_PATH) as conn:
        conn.execute(
            "INSERT INTO rag_manual (judul, penulis, penerbit, sumber) VALUES (?, ?, ?, ?)",
            (judul, penulis, penerbit, sumber),
        )
        conn.commit()
    print(f"Tersimpan ke RAG manual: {judul} - {penerbit}")


def cari_di_rag_manual(judul: str, penulis: str, ambang_batas: float = 0.75):
    """Cari buku yang MIRIP (bukan harus persis sama) di basis RAG manual,
    pakai fuzzy string matching (difflib, bawaan Python). ambang_batas
    0.0-1.0, makin tinggi makin ketat kemiripan yang diterima."""
    with sqlite3.connect(CACHE_DB_PATH) as conn:
        semua = conn.execute("SELECT judul, penulis, penerbit FROM rag_manual").fetchall()

    if not semua:
        return None

    query_gabungan = f"{judul or ''} {penulis or ''}".lower()

    kandidat_terbaik = None
    skor_terbaik = 0.0

    for j, p, penerbit in semua:
        target_gabungan = f"{j or ''} {p or ''}".lower()
        skor = difflib.SequenceMatcher(None, query_gabungan, target_gabungan).ratio()
        if skor > skor_terbaik:
            skor_terbaik = skor
            kandidat_terbaik = penerbit

    if skor_terbaik >= ambang_batas:
        return kandidat_terbaik
    return None


def lengkapi_metadata(judul: str, penulis: str) -> dict:
    if not judul:
        return {"penerbit": None, "sumber": None, "isbn": None}

    key = _normalisasi_key(judul, penulis)

    with sqlite3.connect(CACHE_DB_PATH) as conn:
        row = conn.execute(
            "SELECT penerbit, sumber, isbn FROM cache_metadata WHERE judul_penulis_key = ?", (key,)
        ).fetchone()

        if row and row[0]:
            isbn = row[2]
            if not isbn:
                baris_katalog = conn.execute(
                    "SELECT isbn FROM katalog WHERE judul = ? AND penulis = ? AND isbn IS NOT NULL LIMIT 1",
                    (judul, penulis)
                ).fetchone()
                isbn = baris_katalog[0] if baris_katalog else None
            return {"penerbit": row[0], "sumber": row[1] + "_cache", "isbn": isbn}

    penerbit_rag = cari_di_rag_manual(judul, penulis)
    if penerbit_rag:
        sumber = "rag_manual"
        penerbit = penerbit_rag
    else:
        penerbit = _ambil_dari_open_library(judul, penulis or "")
        sumber = "open_library" if penerbit else None

    if penerbit:
        with sqlite3.connect(CACHE_DB_PATH) as conn:
            conn.execute(
                "INSERT OR REPLACE INTO cache_metadata (judul_penulis_key, penerbit, sumber, isbn) VALUES (?, ?, ?, ?)",
                (key, penerbit, sumber, None),
            )
            conn.commit()

    return {"penerbit": penerbit, "sumber": sumber, "isbn": None}

    # BARU: kalau Open Library gagal total, coba cari di basis RAG
    # manual sebelum menyerah jadi null
    if not penerbit:
        penerbit = cari_di_rag_manual(judul, penulis)
        sumber = "rag_manual" if penerbit else None
    else:
        sumber = "open_library"

    if penerbit:
        with sqlite3.connect(CACHE_DB_PATH) as conn:
            conn.execute(
                "INSERT OR REPLACE INTO cache_metadata (judul_penulis_key, penerbit, sumber, isbn) VALUES (?, ?, ?, ?)",
                (key, penerbit, sumber, None),
            )
            conn.commit()

    return {"penerbit": penerbit, "sumber": sumber, "isbn": None}


def _simpan_ke_katalog(hasil: dict) -> int:
    conn = sqlite3.connect(CACHE_DB_PATH)
    cursor = conn.execute(
        """INSERT INTO katalog (judul, penulis, penerbit, penerbit_sumber, isbn, kategori, status_konfirmasi)
           VALUES (?, ?, ?, ?, ?, ?, ?)""",
        (hasil.get("judul"), hasil.get("penulis"), hasil.get("penerbit"),
         hasil.get("_penerbit_sumber"), hasil.get("isbn"), hasil.get("kategori"), "otomatis"),
    )
    conn.commit()
    id_baru = cursor.lastrowid
    conn.close()
    return id_baru


def _bandingkan_penerbit(a: str, b: str) -> bool:
    def bersih(s):
        s = (s or "").strip().lower().replace("&", "and")
        return re.sub(r"[^a-z0-9]+", " ", s).strip()
    a_bersih, b_bersih = bersih(a), bersih(b)
    if a_bersih == b_bersih:
        return True
    if a_bersih and b_bersih:
        return a_bersih in b_bersih or b_bersih in a_bersih
    return False


def _simpan_atau_gabung_katalog(hasil: dict) -> int:
    key_baru = _normalisasi_key(hasil.get("judul"), hasil.get("penulis"))

    conn = sqlite3.connect(CACHE_DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute("SELECT id, judul, penulis, penerbit, isbn FROM katalog").fetchall()

    for row in rows:
        if _normalisasi_key(row["judul"], row["penulis"]) != key_baru:
            continue

        lama = row["penerbit"]
        baru = hasil.get("penerbit")
        cocok = (not lama) or (not baru) or _bandingkan_penerbit(lama, baru)

        if cocok:
            if hasil.get("isbn") and not row["isbn"]:
                conn.execute("UPDATE katalog SET isbn = ? WHERE id = ?", (hasil["isbn"], row["id"]))
            if hasil.get("penerbit") and not row["penerbit"]:
                conn.execute("UPDATE katalog SET penerbit = ? WHERE id = ?", (hasil["penerbit"], row["id"]))
            conn.execute("UPDATE katalog SET stok = stok + 1 WHERE id = ?", (row["id"],))
            conn.commit()
            conn.close()
            return row["id"]

    conn.close()
    return _simpan_ke_katalog(hasil)


def lihat_katalog(hanya_belum_dikonfirmasi: bool = False) -> list:
    conn = sqlite3.connect(CACHE_DB_PATH)
    conn.row_factory = sqlite3.Row
    query = "SELECT * FROM katalog"
    if hanya_belum_dikonfirmasi:
        query += " WHERE status_konfirmasi = 'otomatis'"
    rows = conn.execute(query).fetchall()
    conn.close()
    return [dict(r) for r in rows]

In [ ]:
import pandas as pd
df = pd.DataFrame(lihat_katalog())
df["key_normalisasi"] = df.apply(lambda r: _normalisasi_key(r["judul"], r["penulis"]), axis=1)
duplikat = df[df.duplicated("key_normalisasi", keep=False)]
print(duplikat[["id", "judul", "penulis", "key_normalisasi"]])

Empty DataFrame
Columns: [id, judul, penulis, key_normalisasi]
Index: []


In [ ]:
import pandas as pd
df_katalog = pd.DataFrame(lihat_katalog())
df_katalog

,id,judul,penulis,penerbit,penerbit_sumber,stok,status_konfirmasi,waktu_masuk,isbn,kategori
0,17,How to Win Friends & Influence People,Dale Carnegie,Simon And Schuster,open_library,13,otomatis,2026-09-14 04:18:35,9781982171452,Self-help
1,19,El Principito,Antoine de Saint-Exupéry,CreateSpace Independent Publishing Platform,open_library,3,otomatis,2026-09-14 07:20:32,9781533622969,fiksi
2,21,Pulang,Leila S. Chudori,Kepustakaan Populer Gramedia,open_library,4,otomatis,2026-09-14 08:29:54,9789799105158,fiksi
3,22,Pride and Prejudice,Jane Austen,FINE EDITIONS PRESS,open_library,1,otomatis,2026-09-15 05:18:33,None,fiksi
4,25,Fahrenheit 451,Ray Bradbury,Simon & Schuster Paperbacks,open_library_isbn,2,otomatis,2026-09-15 08:58:06,9781451673319,None
5,28,Laskar Pelangi,Andrea Hirata,Bentang Pustaka,rag_manual,1,otomatis,2026-09-16 08:51:04,None,fiksi
6,29,Cantik Itu Luka,Eka Kurniawan,Text Publishing Company,open_library_cache,1,otomatis,2026-09-16 08:57:50,None,fiksi


In [ ]:
conn = sqlite3.connect(CACHE_DB_PATH)
conn.execute("UPDATE katalog SET kategori = 'fiksi', stok = 1 WHERE id = 27")
conn.execute("DELETE FROM katalog WHERE id = 27")
conn.commit()
conn.close()

In [ ]:
conn = sqlite3.connect(CACHE_DB_PATH)
conn.execute("DELETE FROM cache_scan")
conn.execute("DELETE FROM cache_metadata")
conn.execute("DELETE FROM katalog")
conn.commit()
conn.close()

In [ ]:
tambah_ke_rag_manual("Cantik Itu Luka", "Eka Kurniawan", "Gramedia Pustaka Utama")
tambah_ke_rag_manual("Pulang", "Leila S. Chudori", "Kepustakaan Populer Gramedia")
tambah_ke_rag_manual("Laskar Pelangi", "Andrea Hirata", "Bentang Pustaka")
tambah_ke_rag_manual("Bumi Manusia", "Pramoedya Ananta Toer", "Lentera Dipantara")
tambah_ke_rag_manual("Ayat-Ayat Cinta", "Habiburrahman El Shirazy", "Republika")
tambah_ke_rag_manual("Negeri 5 Menara", "Ahmad Fuadi", "Gramedia Pustaka Utama")
tambah_ke_rag_manual("Perahu Kertas", "Dee Lestari", "Bentang Pustaka")
tambah_ke_rag_manual("Ronggeng Dukuh Paruk", "Ahmad Tohari", "Gramedia Pustaka Utama")
tambah_ke_rag_manual("Sang Pemimpi", "Andrea Hirata", "Bentang Pustaka")
tambah_ke_rag_manual("Rindu", "Tere Liye", "Republika")
tambah_ke_rag_manual("Filosofi Teras", "Henry Manampiring", "Penerbit Buku Kompas")

Tersimpan ke RAG manual: Cantik Itu Luka - Gramedia Pustaka Utama
Tersimpan ke RAG manual: Pulang - Kepustakaan Populer Gramedia
Tersimpan ke RAG manual: Laskar Pelangi - Bentang Pustaka
Tersimpan ke RAG manual: Bumi Manusia - Lentera Dipantara
Tersimpan ke RAG manual: Ayat-Ayat Cinta - Republika
Tersimpan ke RAG manual: Negeri 5 Menara - Gramedia Pustaka Utama
Tersimpan ke RAG manual: Perahu Kertas - Bentang Pustaka
Tersimpan ke RAG manual: Ronggeng Dukuh Paruk - Gramedia Pustaka Utama
Tersimpan ke RAG manual: Sang Pemimpi - Bentang Pustaka
Tersimpan ke RAG manual: Rindu - Republika
Tersimpan ke RAG manual: Filosofi Teras - Penerbit Buku Kompas


In [ ]:
import pandas as pd

conn = sqlite3.connect(CACHE_DB_PATH)
df_rag = pd.read_sql_query("SELECT * FROM rag_manual", conn)
conn.close()

df_rag

,id,judul,penulis,penerbit,sumber,waktu_masuk
0,1,Pulang,Leila S. Chudori,Kepustakaan Populer Gramedia,input_manual,2026-09-16 08:17:47
1,2,Cantik Itu Luka,Eka Kurniawan,Gramedia Pustaka Utama,input_manual,2026-09-16 08:29:52
2,3,Pulang,Leila S. Chudori,Kepustakaan Populer Gramedia,input_manual,2026-09-16 08:29:52
3,4,Laskar Pelangi,Andrea Hirata,Bentang Pustaka,input_manual,2026-09-16 08:29:52
4,5,Bumi Manusia,Pramoedya Ananta Toer,Lentera Dipantara,input_manual,2026-09-16 08:29:52
5,6,Ayat-Ayat Cinta,Habiburrahman El Shirazy,Republika,input_manual,2026-09-16 08:29:52
6,7,Negeri 5 Menara,Ahmad Fuadi,Gramedia Pustaka Utama,input_manual,2026-09-16 08:29:52
7,8,Perahu Kertas,Dee Lestari,Bentang Pustaka,input_manual,2026-09-16 08:29:52
8,9,Ronggeng Dukuh Paruk,Ahmad Tohari,Gramedia Pustaka Utama,input_manual,2026-09-16 08:29:52
9,10,Sang Pemimpi,Andrea Hirata,Bentang Pustaka,input_manual,2026-09-16 08:29:52


## 8. Jalur ISBN (tanpa Qwen sama sekali)



In [ ]:
def _ambil_dari_open_library(judul: str, penulis: str):
    headers = {"User-Agent": "ScanBukuPerpus/1.0 (contoh@sekolah.sch.id)"}
    BAHASA_PRIORITAS = {"eng", "ind"}

    query = judul
    if penulis:
        query += f" {penulis}"
    search_url = (
        f"https://openlibrary.org/search.json?q={requests.utils.quote(query)}"
        f"&fields=title,author_name,publisher,key&limit=5"
    )

    try:
        res = requests.get(search_url, headers=headers, timeout=15)
        if res.status_code != 200:
            print(f"[DEBUG] Open Library status {res.status_code}")
            return None
        docs = res.json().get("docs", [])
    except Exception as e:
        print(f"[DEBUG] Open Library error: {e}")
        return None

    if not docs:
        return None

    work_key = docs[0].get("key")
    publisher_agregat_fallback = None
    if docs[0].get("publisher"):
        publisher_agregat_fallback = docs[0]["publisher"][0]

    if not work_key:
        return publisher_agregat_fallback

    try:
        ed_url = f"https://openlibrary.org{work_key}/editions.json?limit=20"
        res = requests.get(ed_url, headers=headers, timeout=15)
        if res.status_code != 200:
            return publisher_agregat_fallback
        entries = res.json().get("entries", [])
    except Exception as e:
        print(f"[DEBUG] Open Library editions error: {e}")
        return publisher_agregat_fallback

    kandidat_lain = []
    for ed in entries:
        pub = ed.get("publishers")
        if not pub:
            continue
        lang_keys = [l.get("key", "").split("/")[-1] for l in ed.get("languages", [])]
        if any(l in BAHASA_PRIORITAS for l in lang_keys):
            return pub[0]
        kandidat_lain.append(pub[0])

    if kandidat_lain:
        return kandidat_lain[0]

    if publisher_agregat_fallback:
        print(f"[DEBUG] Tidak ada edisi eng/ind dengan publisher untuk {work_key}, pakai fallback agregat")
    return publisher_agregat_fallback

## 9. Konfirmasi Petugas (opsional - untuk koreksi data)



In [ ]:
def update_status_konfirmasi(id_katalog: int, koreksi: dict = None, stok: int = None) -> dict:
    """
    Update entry yang SUDAH ada di katalog (bukan insert baru - insert
    sudah terjadi otomatis dari ask_model). Dipakai petugas untuk
    mengoreksi field yang salah, atau menandai sudah direview.

    id_katalog : ambil dari hasil["_id_katalog"] setelah ask_model()
    koreksi    : dict field yang mau ditimpa, misal {"penerbit": "Gramedia"}
    stok       : jumlah eksemplar fisik buku ini
    """
    conn = sqlite3.connect(CACHE_DB_PATH)

    if koreksi:
        for field, value in koreksi.items():
            if field in ("judul", "penulis", "penerbit"):
                conn.execute(f"UPDATE katalog SET {field} = ? WHERE id = ?", (value, id_katalog))

    if stok is not None:
        conn.execute("UPDATE katalog SET stok = ? WHERE id = ?", (stok, id_katalog))

    conn.execute("UPDATE katalog SET status_konfirmasi = ? WHERE id = ?", ("terkonfirmasi", id_katalog))
    conn.commit()
    conn.close()

    print(f"Katalog id={id_katalog} sudah diupdate & ditandai terkonfirmasi.")


# Contoh pakai jalur scan:
# hasil = ask_model("/path/ke/cover.webp")
# print(hasil)  # sudah otomatis masuk katalog, cek hasil["_id_katalog"]
# update_status_konfirmasi(hasil["_id_katalog"], koreksi={"penerbit": "Gramedia"}, stok=3)

# Contoh pakai jalur ISBN (perlu simpan manual karena cari_dari_isbn
# belum otomatis masuk katalog seperti ask_model):
# hasil = cari_dari_isbn("9780385533225")
# id_baru = _simpan_ke_katalog(hasil)
# update_status_konfirmasi(id_baru, stok=2)

In [ ]:
   conn = sqlite3.connect(CACHE_DB_PATH)
   conn.execute("DELETE FROM cache_metadata")  # hapus semua, mulai bersih dari nol
   conn.commit()
   conn.close()

## 10. Live API (opsional)


In [ ]:
from fastapi import FastAPI, UploadFile, File, Header, HTTPException, Request
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import shutil
import time
from collections import defaultdict
from PIL import Image
import io

NGROK_AUTHTOKEN = ""
API_KEY = ""
ngrok.set_auth_token(NGROK_AUTHTOKEN)

app = FastAPI()

RATE_LIMIT_MAX = 30
RATE_LIMIT_WINDOW = 60

_riwayat_request = defaultdict(list)  # {ip: [timestamp1, timestamp2, ...]}


def _cek_rate_limit(ip: str) -> bool:
    """True kalau MASIH BOLEH request, False kalau sudah kelebihan limit."""
    sekarang = time.time()
    _riwayat_request[ip] = [t for t in _riwayat_request[ip] if sekarang - t < RATE_LIMIT_WINDOW]

    if len(_riwayat_request[ip]) >= RATE_LIMIT_MAX:
        return False

    _riwayat_request[ip].append(sekarang)
    return True


def _resize_dan_konversi(file_bytes: bytes, max_size: int = 1280) -> str:
    img = Image.open(io.BytesIO(file_bytes)).convert("RGB")
    lebar, tinggi = img.size
    if max(lebar, tinggi) > max_size:
        rasio = max_size / max(lebar, tinggi)
        img = img.resize((int(lebar * rasio), int(tinggi * rasio)), Image.LANCZOS)
    temp_path = f"/tmp/scan_{hash(file_bytes)}.jpg"
    img.save(temp_path, "JPEG", quality=88)
    return temp_path


@app.post("/scan")
async def scan_endpoint(request: Request, file: UploadFile = File(...), x_api_key: str = Header(None)):
    ip_pengirim = request.client.host
    if not _cek_rate_limit(ip_pengirim):
        raise HTTPException(status_code=429, detail="Terlalu banyak request, coba lagi sebentar")

    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="API key salah")

    file_bytes = await file.read()
    temp_path = _resize_dan_konversi(file_bytes)

    hasil = ask_model(temp_path)
    os.remove(temp_path)
    return hasil


@app.get("/isbn/{isbn}")
async def isbn_endpoint(isbn: str, request: Request, x_api_key: str = Header(None)):
    ip_pengirim = request.client.host
    if not _cek_rate_limit(ip_pengirim):
        raise HTTPException(status_code=429, detail="Terlalu banyak request, coba lagi sebentar")

    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="API key salah")
    return cari_dari_isbn(isbn)


nest_asyncio.apply()
public_url = ngrok.connect(8000, domain="closable-bullish-showman.ngrok-free.dev")
print("Live API jalan di:", public_url)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

Live API jalan di: NgrokTunnel: "https://closable-bullish-showman.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [891]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     103.84.230.106:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     103.84.230.106:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     103.84.230.106:0 - "POST /scan HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [891]


In [ ]:
import secrets; secrets.token_urlsafe(32)


'e-oLL-jeWI5TQ6Mhsj38KParg-_r72ccqxTNPSx8A-g'

## 11. Mode Batch (admin upload banyak cover sekaligus)

.

In [ ]:
def scan_folder_batch(folder_path: str, output_json: str = "hasil_scan_batch.json"):
    image_files = sorted([
        f for f in os.listdir(folder_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    ])

    hasil_batch = []
    for fname in image_files:
        fpath = os.path.join(folder_path, fname)
        hasil = ask_model(fpath)
        hasil["nama_file"] = fname
        hasil_batch.append(hasil)
        tanda_cache = " (dari cache)" if hasil.get("_dari_cache_scan") else ""
        print(f"[{fname}]{tanda_cache} judul: {hasil.get('judul')!r} | penerbit: {hasil.get('penerbit')!r}")

    out_path = os.path.join(folder_path, output_json)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(hasil_batch, f, ensure_ascii=False, indent=2)

    print(f"\nTersimpan {len(hasil_batch)} draft ke {out_path}")
    return hasil_batch


# Contoh pakai:
# hasil_draft = scan_folder_batch(FOLDER_PATH)